# Protótipo 3 — LSTM para a seta de tendência futura (15 min)

**Pergunta:** olhando só as 12 últimas glicemias, a glicemia vai **descer, ficar estável ou subir** nos próximos 15 minutos?

**Na prática:** a LSTM prevê o número `delta15 = G(t+15) − G(t)`. Esse número vira uma seta (↓ → ↑) com limites fixos de ±15 mg/dL. Na avaliação comparamos a seta prevista com a variação que **realmente aconteceu**. O futuro é o observado, com tudo o que ocorreu nesse intervalo (refeições, insulina…); não há cenário "sem refeição".

Entradas: **somente CGM** (sem horário, participante, carboidratos ou insulina). Comparamos com dois baselines: **sempre estável** e **continuação da tendência recente**.

`QUICK` substitui o `--quick` do script.

## 0. Configuração

In [ ]:
import os, re, glob, json, time, platform, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, recall_score, accuracy_score, confusion_matrix
warnings.filterwarnings("ignore")

# ----------------------------- CONFIGURAÇÃO -----------------------------
QUICK = True                 # True = execução pequena (rápida); False = execução completa
SEED = 42
SUBJECTS_WANTED = [1, 2, 3]  # None = todos os participantes encontrados
DATA_DIR = "/content/drive/MyDrive/AZT1D 2025/CGM Records"   # <-- ajuste para a sua pasta

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_OUT = "/content/drive/MyDrive"
except ImportError:          # rodando fora do Colab
    BASE_OUT = "."
OUT_DIR = os.path.join(BASE_OUT, "results", "prototipo_03_seta")
os.makedirs(OUT_DIR, exist_ok=True)
print("QUICK =", QUICK, "| DATA_DIR =", DATA_DIR, "| OUT_DIR =", OUT_DIR)

## 1. Preparação (somente glicemia)

Grade de 5 min `(t−5, t]` rotulada no final, CGM sem interpolação, intervalos com valores conflitantes viram ausentes, e exigimos **segmento contínuo** da primeira entrada até o alvo. Como cada participante é processado separadamente, nunca emendamos o fim da série de uma pessoa com o começo de outra. (As funções ainda leem carboidrato/bolus por reaproveitamento, mas **esses campos não entram em nenhuma entrada**.)

In [ ]:
def find_subject_files(data_dir):
    """Localiza os CSVs e identifica o participante pelo nome da pasta/arquivo (ex.: 'Subject 3')."""
    found = {}
    for f in glob.glob(os.path.join(data_dir, "**", "*.csv"), recursive=True):
        m = re.search(r"Subject\s*_?(\d+)", f, flags=re.I)
        if m:
            found.setdefault(int(m.group(1)), f)
    return dict(sorted(found.items()))


def prepare_subject(path):
    """CSV bruto -> grade regular de 5 min com colunas cgm, carb, bolus. Devolve (grade, auditoria)."""
    audit = {}
    df = pd.read_csv(path)
    audit["linhas_brutas"] = len(df)

    # Alias ambíguo: sem coluna CGM não dá para garantir que os registros são CGM -> exclui
    if "CGM" not in df.columns:
        return None, {**audit, "excluido": "sem coluna 'CGM' (alias 'Readings (CGM / BGM)' não confirmado)"}

    n0 = len(df); df = df.drop_duplicates(); audit["duplicadas_removidas"] = n0 - len(df)
    df["EventDateTime"] = pd.to_datetime(df["EventDateTime"], errors="coerce")
    audit["datas_invalidas"] = int(df["EventDateTime"].isna().sum())
    df = df.dropna(subset=["EventDateTime"]).sort_values("EventDateTime")

    for c in ["CGM", "CarbSize", "TotalBolusInsulinDelivered"]:
        if c not in df.columns:
            df[c] = np.nan; audit[f"{c}_coluna_ausente"] = True
        raw = df[c]; num = pd.to_numeric(raw, errors="coerce")
        audit[f"{c}_nao_numerico"] = int((num.isna() & raw.notna()).sum())
        audit[f"{c}_ausente"] = int(raw.isna().sum())
        df[c] = num

    df.loc[df["CGM"] <= 0, "CGM"] = np.nan                       # glicemia inválida -> ausente (nunca zero)
    df.loc[df["CarbSize"] < 0, "CarbSize"] = np.nan
    df.loc[df["TotalBolusInsulinDelivered"] < 0, "TotalBolusInsulinDelivered"] = np.nan

    # Intervalo (t-5min, t] rotulado em t  ==  arredondar PARA CIMA para múltiplos de 5 min
    df["bin"] = df["EventDateTime"].dt.ceil("5min")
    g = df.groupby("bin")

    mm = g["CGM"].agg(["min", "max"])
    cgm = mm["min"].where(mm["min"] == mm["max"])                 # 1 único valor distinto -> mantém; conflito -> NaN
    audit["intervalos_cgm_conflitantes"] = int(((mm["min"] != mm["max"]) & mm["min"].notna()).sum())
    carb = g["CarbSize"].sum()                                    # soma dos registros distintos; NaN é ignorado
    bol = g["TotalBolusInsulinDelivered"].sum()

    idx = pd.date_range(df["bin"].min(), df["bin"].max(), freq="5min")
    grid = pd.DataFrame({"cgm": cgm.reindex(idx),
                         "carb": carb.reindex(idx).fillna(0.0),   # 0 = nenhum evento REGISTRADO
                         "bolus": bol.reindex(idx).fillna(0.0)}, index=idx)
    audit["passos_na_grade"] = len(grid)
    audit["passos_cgm_ausente"] = int(grid["cgm"].isna().sum())
    return grid, audit


def make_cuts(n):
    """60% treino / 20% validação / 20% teste, em ordem cronológica, como (início, fim) de índices da grade."""
    a, b = int(n * 0.6), int(n * 0.8)
    return {"treino": (0, a), "validacao": (a, b), "teste": (b, n)}


def valid_origins(cgm, part, back_cgm, back_ev, fwd):
    """Índices de origem t válidos dentro de UMA partição.
    - CGM sem nenhuma lacuna de t-back_cgm até t+fwd (segmento contínuo até o alvo);
    - todo o suporte (inclusive a janela de eventos back_ev e o alvo) dentro da partição."""
    s, e = part
    ok = ~np.isnan(cgm)
    cs = np.concatenate([[0], np.cumsum(ok)])
    i = np.arange(s + max(back_cgm, back_ev), e - fwd)
    lo, hi = i - back_cgm, i + fwd
    return i[(cs[hi + 1] - cs[lo]) == (hi - lo + 1)]


def prefix(a):
    """Soma acumulada com zero na frente: soma dos passos a..b = p[b+1] - p[a]."""
    return np.concatenate([[0.0], np.cumsum(a)])


def first_run(idx, min_len):
    """Primeiro trecho de origens consecutivas (passo 1 da grade) com pelo menos min_len pontos."""
    if len(idx) == 0: return None
    breaks = np.where(np.diff(idx) != 1)[0] + 1
    for run in np.split(np.arange(len(idx)), breaks):
        if len(run) >= min_len: return run
    return None


def md_table(df, nd=2):
    cols = list(df.columns)
    L = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        L.append("| " + " | ".join(f"{v:.{nd}f}" if isinstance(v, (float, np.floating)) else str(v) for v in r) + " |")
    return "\n".join(L)


def mae_rmse(y, p):
    if len(y) == 0: return np.nan, np.nan
    e = np.asarray(p) - np.asarray(y)
    return float(np.mean(np.abs(e))), float(np.sqrt(np.mean(e ** 2)))

In [ ]:
files = find_subject_files(DATA_DIR)
print("Participantes encontrados:", list(files))
if SUBJECTS_WANTED is not None:
    files = {k: v for k, v in files.items() if k in SUBJECTS_WANTED}
assert files, "Nenhum CSV encontrado: confira DATA_DIR (e se o Drive foi montado)."

t0 = time.time()
grids, audits = {}, {}
for sid, path in files.items():
    g, a = prepare_subject(path)
    audits[sid] = a
    if g is not None: grids[sid] = g
t_prep_load = time.time() - t0

pd.DataFrame(audits).T

## 2. Divisão temporal

60% / 20% / 20% cronológico por participante; janelas geradas depois, com entrada e alvo na mesma partição.

In [ ]:
cuts, cut_times = {}, {}
for sid, g in grids.items():
    cuts[sid] = make_cuts(len(g))
    cut_times[sid] = {p: [str(g.index[s]), str(g.index[e - 1])] for p, (s, e) in cuts[sid].items()}
pd.DataFrame({sid: {p: f"{v[0]} -> {v[1]}" for p, v in d.items()} for sid, d in cut_times.items()}).T

## 3. Janelas, alvo e setas

Na grade de 5 min, 15 min = **3 passos** (não 15!):

```text
Entrada: G(t−55) … G(t)  = cgm[i-11 … i]      (a última é G(t), nunca G(t+5))
Alvo:    delta15 = G(t+15) − G(t) = cgm[i+3] − cgm[i]
```

| Variação em 15 min | Classe | Símbolo |
|---|---|---|
| < −15 mg/dL | Descida | ↓ |
| entre −15 e +15 (inclusive) | Estável | → |
| > +15 mg/dL | Subida | ↑ |

É uma convenção operacional deste protótipo (±1 mg/dL/min), **não** a seta de um fabricante nem uma classificação clínica validada; os limites não são ajustados olhando o teste. A seta resume o **saldo** até +15 min: não garante trajetória monotônica nem detecta picos intermediários.

In [ ]:
CLASSES = ["descida", "estável", "subida"]
SYM = {0: "↓", 1: "→", 2: "↑"}
THR = 15.0


def to_class(delta):
    d = np.asarray(delta, dtype=float)
    return np.where(d < -THR, 0, np.where(d > THR, 2, 1))

# limites: exatamente -15 e +15 são "estável"
assert to_class([-15.0001, -15.0, 0.0, 15.0, 15.0001]).tolist() == [0, 1, 1, 1, 2]

BACK, FWD = 11, 3
def build_p3(sid, grid, cuts_sid):
    cgm = grid["cgm"].values
    parts = {}
    for part, rng in cuts_sid.items():
        i = valid_origins(cgm, rng, BACK, 0, FWD)
        if len(i) == 0: continue
        X = np.stack([cgm[i - k] for k in range(11, -1, -1)], axis=1)       # (n, 12)
        y = cgm[i + FWD] - cgm[i]                                            # delta real em 15 min
        trend = cgm[i] - cgm[i - 3]                                          # G(t) - G(t-15)
        meta = pd.DataFrame({"participante": sid, "origem_idx": i, "origem": grid.index[i], "alvo_t": grid.index[i + FWD],
                             "g_atual": cgm[i], "g_futura": cgm[i + FWD], "delta_real": y, "delta_tendencia": trend})
        parts[part] = (X, y, meta)
    return parts

built, discarded = {}, {}
for sid, g in grids.items():
    r = build_p3(sid, g, cuts[sid])
    if set(r) != {"treino", "validacao", "teste"}: discarded[sid] = "sem exemplos nas três partições"
    else: built[sid] = r
eligible = sorted(built)
if QUICK: eligible = eligible[:3]
print("Elegíveis:", eligible, "| descartados:", discarded)

D = {p: (np.concatenate([built[s][p][0] for s in eligible]),
         np.concatenate([built[s][p][1] for s in eligible]),
         pd.concat([built[s][p][2] for s in eligible], ignore_index=True)) for p in ["treino", "validacao", "teste"]}

# verificações: alvo a +15 min; partições respeitadas
for p in D: assert ((D[p][2]["alvo_t"] - D[p][2]["origem"]) == pd.Timedelta("15min")).all()
for s in eligible:
    for p, (s0, e0) in cuts[s].items():
        idx = built[s][p][2]["origem_idx"]; assert idx.min() - BACK >= s0 and idx.max() + FWD <= e0 - 1
pd.DataFrame({p: {f"Subject {s}": len(built[s][p][1]) for s in eligible} for p in D})

In [ ]:
# Distribuição das classes reais por partição (uma frequência alta de "estável" é esperada)
dist = pd.DataFrame({p: pd.Series(to_class(D[p][1])).map(dict(enumerate(CLASSES))).value_counts() for p in D}).reindex(CLASSES).fillna(0).astype(int)
dist

## 4. Escalas e modelo

Escala de CGM (média/desvio global do treino) e do delta (padronização no treino). A rede é mínima:

```text
(12×1) → LSTM(32) → último estado → Linear(32, 1)  →  delta padronizado
```

MSE sobre o delta padronizado, Adam `lr=0.001`, lote 64, semente 42, até 15 épocas (3 no rápido), parada após 3 sem melhora da validação, restaurando o melhor estado. Sem cabeça de classificação, sem balanceamento, sem busca de parâmetros. No modo rápido o treino é amostrado em até 20.000 janelas (semente 42).

In [ ]:
Xtr, ytr, mtr = D["treino"]
if QUICK and len(Xtr) > 20000:
    sel = np.sort(np.random.default_rng(SEED).choice(len(Xtr), 20000, replace=False))
    Xtr, ytr = Xtr[sel], ytr[sel]
print("Janelas de treino usadas:", len(Xtr))

cgm_mu, cgm_sd = float(Xtr.mean()), float(Xtr.std())
y_sc = StandardScaler().fit(ytr.reshape(-1, 1))

def tens(X, y):
    return (torch.tensor(((X - cgm_mu) / cgm_sd)[..., None], dtype=torch.float32),
            torch.tensor(y_sc.transform(y.reshape(-1, 1)), dtype=torch.float32))

Ttr = tens(Xtr, ytr); Tva = tens(D["validacao"][0], D["validacao"][1])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.out(h[-1])


@torch.no_grad()
def predict_scaled(net, x, bs=1024):
    net.eval()
    return torch.cat([net(x[k:k+bs].to(device)).cpu() for k in range(0, len(x), bs)]).numpy()


def train_net(max_epochs, patience=3):
    torch.manual_seed(SEED); np.random.seed(SEED)
    net = Net().to(device); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    gen = torch.Generator().manual_seed(SEED)
    x, y = Ttr; n = len(x)
    best, best_state, bad, hist = np.inf, None, 0, []
    for ep in range(max_epochs):
        net.train(); perm = torch.randperm(n, generator=gen)        # embaralha só o treino
        for k in range(0, n, 64):
            b = perm[k:k + 64]; opt.zero_grad()
            loss = nn.functional.mse_loss(net(x[b].to(device)), y[b].to(device)); loss.backward(); opt.step()
        v = float(nn.functional.mse_loss(torch.tensor(predict_scaled(net, Tva[0])), Tva[1]))
        hist.append(v); print(f"época {ep+1}: perda de validação = {v:.4f}")
        if v < best - 1e-9:
            best, bad = v, 0; best_state = {k: t.detach().cpu().clone() for k, t in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(best_state)                                  # melhor estado de validação
    return net, hist

t0 = time.time()
net, hist = train_net(3 if QUICK else 15)
t_train = time.time() - t0
print(f"treino: {t_train:.1f}s")

## 5. Previsores e baselines (mesmas linhas)

1. **Sempre estável:** delta previsto = 0 → seta sempre →.
2. **Tendência recente:** delta futuro previsto = `G(t) − G(t−15)`, convertido nos mesmos limites. É avaliada contra o delta dos **próximos** 15 min, não contra a inclinação passada.
3. **LSTM:** delta da rede, invertendo a escala **antes** de aplicar os limites.

In [ ]:
t0 = time.time()
frames = []
for p in ["validacao", "teste"]:
    X, y, meta = D[p]
    d = meta.copy(); d["particao"] = p
    d["delta_lstm"] = y_sc.inverse_transform(predict_scaled(net, tens(X, y)[0])).ravel()
    d["delta_estavel"] = 0.0
    frames.append(d)
pred = pd.concat(frames, ignore_index=True)
t_infer = time.time() - t0

PRED_COL = {"sempre_estavel": "delta_estavel", "tendencia_recente": "delta_tendencia", "lstm": "delta_lstm"}
pred["classe_real"] = to_class(pred["delta_real"])
for m, col in PRED_COL.items():
    pred[f"classe_{m}"] = to_class(pred[col])
# glicemia futura reconstruída = G(t) + delta previsto
for m, col in PRED_COL.items():
    pred[f"g_futura_prev_{m}"] = pred["g_atual"] + pred[col]
assert pred[[f"classe_{m}" for m in PRED_COL]].notna().all().all()
print(f"inferência: {t_infer:.1f}s")

## 6. Métricas

**Principais:** macro-F1, matriz de confusão e recall de subida/descida. A acurácia aparece, mas nunca sozinha: se "estável" domina, uma seta que quase nunca muda já parece boa.

Regras fixas: classes sempre na ordem `descida, estável, subida`; macro-F1 sobre os **três** rótulos com `zero_division=0`; se a classe não existe no teste de um participante, o recall dela é `NaN` (não estimável). MAE/RMSE do delta (= erro da glicemia futura reconstruída).

In [ ]:
def class_metrics(sub, m):
    yt, yp = sub["classe_real"].values, sub[f"classe_{m}"].values
    sup = [int((yt == k).sum()) for k in range(3)]
    rec = recall_score(yt, yp, labels=[0, 1, 2], average=None, zero_division=0) if len(yt) else [np.nan] * 3
    rec = [r if s > 0 else np.nan for r, s in zip(rec, sup)]            # sem suporte => não estimável
    mae, rmse = mae_rmse(sub["delta_real"], sub[PRED_COL[m]])
    return dict(n=len(sub), acuracia=accuracy_score(yt, yp) if len(yt) else np.nan,
                macro_F1=f1_score(yt, yp, labels=[0, 1, 2], average="macro", zero_division=0) if len(yt) else np.nan,
                recall_descida=rec[0], recall_estavel=rec[1], recall_subida=rec[2],
                n_descida=sup[0], n_estavel=sup[1], n_subida=sup[2], MAE_delta=mae, RMSE_delta=rmse)

rows = []
for p in ["validacao", "teste"]:
    d = pred[pred.particao == p]
    for who, sub in [("AGREGADO", d)] + [(f"Subject {s}", d[d.participante == s]) for s in eligible]:
        for m in PRED_COL:
            rows.append(dict(particao=p, modelo=m, participante=who, **class_metrics(sub, m)))
metricas = pd.DataFrame(rows)
metricas.to_csv(os.path.join(OUT_DIR, "metricas.csv"), index=False)
metricas[(metricas.particao == "teste") & (metricas.participante == "AGREGADO")].set_index("modelo").drop(columns=["particao", "participante"]).round(3)

In [ ]:
# Por participante (teste): macro-F1 e recall de subida/descida
metricas[(metricas.particao == "teste") & (metricas.participante != "AGREGADO")] \
    [["participante", "modelo", "n", "macro_F1", "recall_descida", "recall_subida", "n_descida", "n_estavel", "n_subida"]].round(3)

## 7. Matrizes de confusão (teste, agregado)

Linhas = classe real; colunas = classe prevista.

In [ ]:
t = pred[pred.particao == "teste"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, m in zip(axes, PRED_COL):
    cm = confusion_matrix(t["classe_real"], t[f"classe_{m}"], labels=[0, 1, 2])
    ax.imshow(cm, cmap="Blues")
    for i in range(3):
        for j in range(3): ax.text(j, i, cm[i, j], ha="center", va="center", color="black")
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("previsto"); ax.set_ylabel("real"); ax.set_title(m)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "matriz_confusao.png"), dpi=130); plt.show()

## 8. Exemplo de 2 h no teste

Primeira janela de teste (primeiro participante que tenha) com 24 origens consecutivas. Em cada origem *t* mostramos a CGM e, abaixo, a seta **real** e as **previstas para os próximos 15 min** (LSTM e tendência recente), no mesmo eixo de origens.

In [ ]:
done = False
for sid in eligible:
    d = pred[(pred.participante == sid) & (pred.particao == "teste")].sort_values("origem_idx")
    run = first_run(d["origem_idx"].values, 24)
    if run is None: continue
    w = d.iloc[run[:24]]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.6]})
    a1.plot(w["origem"], w["g_atual"], "k-o", ms=3); a1.set_ylabel("CGM (mg/dL)")
    a1.set_title(f"Subject {sid} — setas na origem t = previsão para t → t+15 min")
    for y, (lab, col) in zip([2, 1, 0], [("real", "classe_real"), ("LSTM", "classe_lstm"), ("tendência", "classe_tendencia_recente")]):
        for _, r in w.iterrows(): a2.text(r["origem"], y, SYM[int(r[col])], ha="center", va="center", fontsize=13)
        a2.text(w["origem"].iloc[0] - pd.Timedelta("9min"), y, lab, ha="right", va="center", fontsize=9)
    a2.set_ylim(-.6, 2.6); a2.set_yticks([]); a2.set_xlim(w["origem"].iloc[0] - pd.Timedelta("30min"), w["origem"].iloc[-1] + pd.Timedelta("6min"))
    plt.xticks(rotation=30); plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "exemplo_15min.png"), dpi=130); plt.show()
    done = True; break
if not done: print("Sem janela contínua de 2 h no teste.")

## 9. Salvar e interpretar

In [ ]:
pred.rename(columns={"origem": "origem_t"}).assign(
    classe_real_nome=lambda d: d.classe_real.map(dict(enumerate(CLASSES)))
).to_csv(os.path.join(OUT_DIR, "previsoes.csv"), index=False)
torch.save(net.state_dict(), os.path.join(OUT_DIR, "pesos_lstm.pt"))
joblib.dump({"cgm_mu": cgm_mu, "cgm_sd": cgm_sd, "delta": y_sc}, os.path.join(OUT_DIR, "escaladores.joblib"))
json.dump(cut_times, open(os.path.join(OUT_DIR, "cortes_temporais.json"), "w"), indent=1, ensure_ascii=False)
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "torch": torch.__version__}
json.dump(dict(QUICK=QUICK, SEED=SEED, participantes=eligible, descartados=discarded, limites_mgdl=THR,
               treino_usado=len(Xtr), epocas=len(hist), versoes=versions),
          open(os.path.join(OUT_DIR, "config.json"), "w"), indent=1, ensure_ascii=False)

a = metricas[(metricas.particao == "teste") & (metricas.participante == "AGREGADO")].set_index("modelo")
f1_l, f1_t, f1_e = a.loc["lstm", "macro_F1"], a.loc["tendencia_recente", "macro_F1"], a.loc["sempre_estavel", "macro_F1"]
concl = [f"Macro-F1 no teste: LSTM {f1_l:.3f} | tendência recente {f1_t:.3f} | sempre estável {f1_e:.3f}.",
         f"Recall de subida/descida da LSTM: {a.loc['lstm','recall_subida']:.3f} / {a.loc['lstm','recall_descida']:.3f} "
         f"(tendência: {a.loc['tendencia_recente','recall_subida']:.3f} / {a.loc['tendencia_recente','recall_descida']:.3f}).",
         "A LSTM superou a tendência recente em macro-F1." if f1_l > f1_t else "A LSTM NÃO superou a tendência recente em macro-F1 (resultado mantido e reportado).",
         f"Acurácia da LSTM {a.loc['lstm','acuracia']:.3f} vs sempre estável {a.loc['sempre_estavel','acuracia']:.3f}: "
         "acurácia isolada não basta, pois a classe estável domina."]
print("\n".join(concl))

resumo = f"""# Resumo — Protótipo 3 (seta de tendência em 15 min)

- Modo: {'rápido' if QUICK else 'completo'} | participantes: {eligible} | descartados: {discarded}
- Distribuição real das classes por partição:\n\n{md_table(dist.reset_index().rename(columns={'index': 'classe'}), 0)}
- Tempos: treino {t_train:.1f}s, inferência {t_infer:.1f}s | épocas: {len(hist)}

## Teste (agregado)

{md_table(a.reset_index()[['modelo','n','acuracia','macro_F1','recall_descida','recall_subida','MAE_delta','RMSE_delta']], 3)}

## Interpretação

""" + "\n".join("- " + c for c in concl) + """

Limites de ±15 mg/dL são convenção operacional, não a seta de um fabricante nem classificação clínica.
A seta é o saldo da mudança até +15 min (sem garantir trajetória monotônica). Avalia pessoas já observadas.
"""
open(os.path.join(OUT_DIR, "resumo.md"), "w").write(resumo)
print(sorted(os.listdir(OUT_DIR)))